<a href="https://colab.research.google.com/github/lilhawkeye2002-ux/Whisper_Notebooks/blob/main/Whisper_Bulk_Transcriber_AltTimestampMethod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Whisper Bulk Transcriber — Timestamped Transcript Variant (Optimized)

Transcribe audio and video files into **millisecond-accurate, timestamped** plain text using OpenAI's Whisper `large-v2` model.

This variant differs from the standard transcriber in one key way: the `.txt` output includes **DTW-aligned timestamps** on every line — sourced directly from when words are spoken in the audio, not rounded token predictions.

---

## What makes this different

| | Standard `Whisper_Bulk_Transcriber` | This notebook |
|---|---|---|
| `.txt` content | Plain text, no timestamps | `[HH:MM:SS.mmm --> HH:MM:SS.mmm]  spoken text` |
| Timestamp source | Model token predictions (20 ms steps) | Dynamic Time Warping on cross-attention weights |
| How accurate | ±20–60 ms | Word-boundary aligned to actual audio |
| `.srt` / `.vtt` | Included | Included (also DTW-refined) |

---

## Performance Tuning Flags

At the top of the code cell you can toggle five flags:

| Flag | Default | What it does | Typical gain |
|---|---|---|---|
| `PRECONVERT_AUDIO` | `True` | Pre-converts every file to 16 kHz mono WAV before transcription, eliminating Whisper's per-chunk resampling overhead. Especially helpful for MP3/AAC/M4A inputs. | 10–30% |
| `CUDNN_BENCHMARK` | `True` | Runs cuDNN's autotuner once to select the fastest convolution kernels for your audio chunk size. Pays off on batches. | small |
| `USE_INFERENCE_MODE` | `True` | Wraps transcription in `torch.inference_mode()` to skip autograd bookkeeping. Free speedup with no quality impact. | 3–10% |
| `FAST_DECODE` | `False` | Switches to greedy decoding (beam\_size=1, temperature=0) with fallback retries disabled. Much faster on clean audio; not recommended for noisy recordings. | up to 2× |
| `TORCH_COMPILE` | `False` | Experimental: `torch.compile(model)` via TorchDynamo/Inductor. Adds ~60s warmup on first call. Gains are inconsistent in Colab. | 5–15% |

### Expected VRAM usage on T4

| Mode | VRAM |
|---|---|
| FP16 (fp16=True, CUDA) | ~3–4 GB model + ~1–2 GB activations |
| FP32 (fp16=False) | ~6–8 GB — **avoid on T4** |

> `fp16` is set automatically based on whether CUDA is available.

---

## How To Use (3 Steps)

### Step 1 — Enable a free GPU
**Runtime → Change runtime type → Hardware accelerator → T4 GPU**

### Step 2 — Run the cell for the first time
Click **▶** on the code cell below. It will create the upload folder and stop with:
> *"Directory created. Upload your audio/video files there, then run this cell again."*

### Step 3 — Upload files and run again
1. Open the **Files panel** (📁 in the left sidebar)
2. Navigate to `/content/bulk_process_audios_here`
3. **Drag and drop** your audio or video files in
4. Click **▶** again — transcription starts automatically

> **Re-running the cell** is safe — if the model is already loaded, it will be reused without reloading.

---

## Output

For each source file, three outputs are saved to `/content/audio_transcription/`:

| Format | Content |
|---|---|
| `.txt` | `[00:00:00.000 --> 00:00:04.820]  Welcome everyone, today we're going to...` |
| `.srt` | Subtitle format with DTW-refined timestamps |
| `.vtt` | Web caption format with DTW-refined timestamps |

All files are zipped to `/content/all_transcribed_files.zip` — **download before closing the session**.

---

## Supported formats
| Audio | Video |
|-------|-------|
| `.mp3` `.wav` `.m4a` `.flac` | `.mp4` `.mov` `.avi` `.mkv` |
| `.aac` `.ogg` `.wma` `.opus` | `.webm` |
| `.aiff` `.amr` `.au` | |

> 🎬 Video files are handled automatically — the audio track is extracted and resampled before transcription.

---

## Troubleshooting

| Issue | Fix |
|---|---|
| Transcription very slow | Switch to GPU: **Runtime → Change runtime type → T4 GPU** |
| `ffmpeg` not found | Add a cell above: `!apt install -y ffmpeg` |
| `CUDA out of memory` on re-run | Model is already loaded — this is now handled automatically. If it persists, do **Runtime → Restart session** and re-run. |
| GPU out of memory (first run) | **Runtime → Restart session**, then re-run |
| `torch.compile` error | Set `TORCH_COMPILE = False` at the top of the cell |
| File skipped `[SKIP]` | File is empty (0 bytes) or unreadable — re-upload it |
| Session expired before download | Re-run the cell — all files are re-transcribed and re-zipped |

In [ ]:
# @title ## Whisper Bulk Transcriber — Timestamped (Optimized)
# @markdown ### Outputs `.txt` with DTW-aligned millisecond timestamps, plus `.srt` and `.vtt`.
# @markdown Final zip saved to `/content/all_transcribed_files.zip`.
# @markdown **Re-running this cell is safe** — the model is reused if already loaded.

# ── Phase 0: Constants & Configuration Flags ──────────────────────────────
# Only stdlib modules needed here — no pip installs required for directory check.

import os
import sys

BULK_DIR   = "/content/bulk_process_audios_here"
OUTPUT_DIR = "/content/audio_transcription"
ZIP_PATH   = "/content/all_transcribed_files.zip"
USE_MODEL  = "large-v2"
DRIVE_BASE = "/content/drive/MyDrive"

VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

SUPPORTED_EXTENSIONS = {
    '.mp3', '.wav', '.m4a', '.flac', '.aac', '.ogg', '.wma',
    '.aiff', '.opus', '.amr', '.au',
} | VIDEO_EXTENSIONS

TRANSCRIBED_EXTS = {'.txt', '.srt', '.vtt'}

# ── Performance Tuning ────────────────────────────────────────────
# Adjust these flags before running to trade speed against quality.

# Pre-convert every audio file to 16 kHz mono WAV before passing it to Whisper.
# Eliminates Whisper's internal ffmpeg resampling overhead on every 30-second
# chunk. Pays off most on compressed formats (MP3, AAC, M4A). Recommended: True.
PRECONVERT_AUDIO   = True  # @param {type:"boolean"}

# Enable cuDNN autotuner. Helps on batches of similarly-sized audio chunks.
CUDNN_BENCHMARK    = True  # @param {type:"boolean"}

# Wrap transcription in torch.inference_mode() — free ~3-10% speedup.
USE_INFERENCE_MODE = True  # @param {type:"boolean"}

# Fast decode mode: greedy decoding (temperature=0, beam_size=1), fallback
# retries disabled. Up to 2x faster on clean speech.
FAST_DECODE        = False  # @param {type:"boolean"}

# Experimental: torch.compile(model). Adds ~60s warmup. Can be unstable in Colab.
TORCH_COMPILE      = False  # @param {type:"boolean"}

# ── Google Drive Integration ──────────────────────────────────────────
#
# AUTO_MOUNT_DRIVE
#   Automatically mount Google Drive before any Drive operations.
#   Default: False (controlled by form checkbox). Auto-enabled when
#   AUTO_IMPORT_FROM_DRIVE is True.
#
# AUTO_IMPORT_FROM_DRIVE
#   Import audio files from a Drive folder into the bulk processing directory
#   before transcription begins. Searches recursively. Default: False.
#
# DRIVE_AUDIO_FOLDER
#   Path relative to MyDrive for the audio import source folder.
#   Example: "home/whatever_folder_/your_audio/is_stored_in"
#
# AUTO_EXPORT_TO_DRIVE
#   Upload the final transcription zip archive to Google Drive after all
#   transcription jobs complete. Default: True.
#
# DRIVE_EXPORT_FOLDER
#   Path relative to MyDrive for the export destination.
#   Fallback 1: DRIVE_AUDIO_FOLDER (if export folder is blank).
#   Fallback 2: warn + skip (if neither is set).

AUTO_MOUNT_DRIVE       = False  # @param {type:"boolean"}
AUTO_IMPORT_FROM_DRIVE = False  # @param {type:"boolean"}
DRIVE_AUDIO_FOLDER     = "home/whatever_folder_/your_audio/is_stored_in"  # @param {type:"string"}
AUTO_EXPORT_TO_DRIVE   = True   # @param {type:"boolean"}
DRIVE_EXPORT_FOLDER    = ""     # @param {type:"string"}

# ── Q24: Auto-enable mount when import is enabled ─────────────────────
# Drive import requires Drive mounted. If AUTO_IMPORT_FROM_DRIVE is enabled
# without AUTO_MOUNT_DRIVE, mount is silently enforced and reported to console.
if AUTO_IMPORT_FROM_DRIVE and not AUTO_MOUNT_DRIVE:
    print("[INFO] AUTO_IMPORT_FROM_DRIVE is enabled — AUTO_MOUNT_DRIVE auto-enabled.")
    AUTO_MOUNT_DRIVE = True

# ── Phase 0b: Google Drive Mount ─────────────────────────────────────
# Executes at the very beginning — before directory check, pip installs, or
# model load — so Drive is available for import/export operations downstream.
#
# Mount method: google.colab.drive.mount() (official Colab API, primary).
# Remount detection: os.path.ismount() avoids unnecessary remounts.
#
# Failure policy (hard stop if mount was requested and fails):
#   Crystal-clear error output with error type, details, and recovery steps.

_DRIVE_MOUNTED = False

if AUTO_MOUNT_DRIVE:
    _drive_mount_path = "/content/drive"
    _already_mounted  = os.path.ismount(_drive_mount_path)

    if _already_mounted:
        print("[Drive] Drive already mounted at /content/drive — skipping remount.")
        _DRIVE_MOUNTED = True
    else:
        print("[Drive] Mounting Google Drive...")
        try:
            from google.colab import drive as _colab_drive
            _colab_drive.mount(_drive_mount_path)
            _DRIVE_MOUNTED = True
            print("[Drive] Mount successful.")
        except Exception as _mount_err:
            _err_type = type(_mount_err).__name__
            _err_msg  = str(_mount_err)
            print(
                f"\n{'!' * 64}\n"
                f"  GOOGLE DRIVE MOUNT FAILED\n"
                f"\n"
                f"  Error type : {_err_type}\n"
                f"  Details    : {_err_msg}\n"
                f"\n"
                f"  What to do:\n"
                f"    1. Ensure you are running in Google Colab (not locally).\n"
                f"    2. Re-run this cell and follow the Drive authorization\n"
                f"       popup that appears. Click 'Allow' when prompted.\n"
                f"    3. If the error persists, try:\n"
                f"         Runtime > Restart session > Re-run cell\n"
                f"    4. To skip Drive mounting and use local file upload instead:\n"
                f"         Set  AUTO_MOUNT_DRIVE        = False\n"
                f"         Set  AUTO_IMPORT_FROM_DRIVE  = False\n"
                f"         Set  AUTO_EXPORT_TO_DRIVE    = False\n"
                f"         Then re-run this cell.\n"
                f"{'!' * 64}\n"
            )
            sys.exit(1)

# ── Phase 0c: Google Drive Audio Import ────────────────────────────
# Imports all supported audio files from a Drive folder into BULK_DIR before
# the directory check runs, so BULK_DIR is populated before validation.
#
# Execution order when drive import is ENABLED:  mount → import → directory check
# Execution order when drive import is DISABLED: directory check → create dir if missing
#
# Collision policy:
#   Same name + same size  → SKIP  (informational log: name, size, reason)
#   Same name + diff size  → OVERWRITE (incoming file wins, logged explicitly)
#
# Transfer verification (industry-standard, no assumptions):
#   Every file is verified by BOTH file size AND SHA-256 hash after copy.
#   The import task does not complete until all verifications pass.
#   Failed copies are cleaned up immediately. Partial files are not left behind.

import hashlib
import shutil


def _file_sha256(path):
    """Compute SHA-256 of a file in 1 MB chunks (safe for large files)."""
    h = hashlib.sha256()
    with open(path, "rb") as _f:
        for _chunk in iter(lambda: _f.read(1 << 20), b""):
            h.update(_chunk)
    return h.hexdigest()


def _copy_with_verification(src, dst):
    """
    Copy src to dst and verify 100% successful transfer via size + SHA-256.

    Copy method priority:
      1. shutil.copy2  (preserves metadata)
      2. shutil.copy   (fallback)
      3. Manual binary read/write loop (final fallback)

    Raises RuntimeError if size or SHA-256 hash mismatch detected.
    Does NOT return until the copy is fully verified complete.
    """
    src_size   = os.path.getsize(src)
    src_sha256 = _file_sha256(src)

    _copied = False
    for _copy_fn, _label in [
        (lambda s, d: shutil.copy2(s, d), "shutil.copy2"),
        (lambda s, d: shutil.copy(s, d),  "shutil.copy"),
    ]:
        try:
            _copy_fn(src, dst)
            _copied = True
            break
        except Exception as _ce:
            print(f"  [WARN] {_label} failed ({_ce}), trying next method...")

    if not _copied:
        # Manual binary copy as final fallback
        with open(src, "rb") as _sf, open(dst, "wb") as _df:
            while True:
                _chunk = _sf.read(1 << 20)
                if not _chunk:
                    break
                _df.write(_chunk)
        print("  [INFO] Used manual binary copy as final fallback.")

    # ── Verify: size ──────────────────────────────────────────
    dst_size = os.path.getsize(dst)
    if dst_size != src_size:
        raise RuntimeError(
            f"Transfer size mismatch — src={src_size:,} bytes, dst={dst_size:,} bytes. "
            f"File may be truncated or corrupt."
        )

    # ── Verify: SHA-256 ─────────────────────────────────────
    dst_sha256 = _file_sha256(dst)
    if dst_sha256 != src_sha256:
        raise RuntimeError(
            f"Transfer hash mismatch (SHA-256):\n"
            f"  Source : {src_sha256}\n"
            f"  Dest   : {dst_sha256}\n"
            f"File is corrupt. Re-run to retry."
        )


def _import_from_drive_folder(drive_folder_path, dest_dir):
    """
    Recursively import all supported audio/video files from drive_folder_path
    into dest_dir. Returns count of files successfully imported.
    """
    os.makedirs(dest_dir, exist_ok=True)

    # ── Validate source folder ────────────────────────────────────
    if not os.path.exists(drive_folder_path):
        print(
            f"\n[Drive Import] The folder path you entered does not exist:\n"
            f"  Path entered : {drive_folder_path}\n"
            f"\n"
            f"  Please check for typos or missing parent folders.\n"
            f"\n"
            f"  To manually import files from Google Drive:\n"
            f"    1. Open the Files panel in Colab (folder icon in left sidebar).\n"
            f"    2. Navigate to /content/bulk_process_audios_here\n"
            f"    3. Drag-and-drop your audio files into that folder.\n"
            f"\n"
            f"  To upload directly from your PC:\n"
            f"    1. In the Files panel, right-click bulk_process_audios_here.\n"
            f"    2. Select 'Upload' and choose your files from your computer.\n"
        )
        _action = input(
            "Choose an option:\n"
            "  [c] Continue without Drive import (use manually uploaded files)\n"
            "  [n] Enter a new Google Drive folder path\n"
            "  [e] Exit — I'll fix the folder path and re-run\n"
            "Your choice [c/n/e]: "
        ).strip().lower()

        if _action == "n":
            _new_path = input(
                "Enter folder path relative to MyDrive\n"
                "(e.g. 'home/whatever_folder_/your_audio/is_stored_in'): "
            ).strip()
            _new_full = os.path.join(DRIVE_BASE, _new_path.lstrip("/"))
            return _import_from_drive_folder(_new_full, dest_dir)
        elif _action == "c":
            print("[Drive Import] Continuing — will use any manually uploaded files.")
            return 0
        else:
            sys.exit(
                "[Drive Import] Exiting. "
                "Update DRIVE_AUDIO_FOLDER to the correct path and re-run."
            )

    # ── Recursively scan for supported audio files ──────────────────────
    source_files = []
    for _root, _dirs, _files in os.walk(drive_folder_path):
        for _fname in sorted(_files):
            if os.path.splitext(_fname)[1].lower() in SUPPORTED_EXTENSIONS:
                source_files.append(os.path.join(_root, _fname))

    if not source_files:
        sys.exit(
            f"\n[Drive Import] No supported audio/video files found in:\n"
            f"  {drive_folder_path}\n"
            f"  (searched recursively including all subdirectories)\n"
            f"  Supported formats: {', '.join(sorted(SUPPORTED_EXTENSIONS))}\n"
            f"Add audio files to that folder and re-run.\n"
        )

    print(f"[Drive Import] Found {len(source_files)} audio file(s) in Drive. Importing...")
    print(f"  Source : {drive_folder_path}")
    print(f"  Dest   : {dest_dir}")
    print()

    _imported  = 0
    _skipped   = 0
    _overwrote = 0
    _failed    = 0

    for _src_path in source_files:
        _fname    = os.path.basename(_src_path)
        _dst_path = os.path.join(dest_dir, _fname)
        _src_size = os.path.getsize(_src_path)

        # ── Collision handling ────────────────────────────────────────
        if os.path.exists(_dst_path):
            _dst_size = os.path.getsize(_dst_path)
            if _dst_size == _src_size:
                print(
                    f"  [SKIP] Duplicate — {_fname}\n"
                    f"         Reason      : Identical file size\n"
                    f"         Source      : {_src_path} ({_src_size:,} bytes)\n"
                    f"         Destination : {_dst_path} ({_dst_size:,} bytes)"
                )
                _skipped += 1
                continue
            else:
                print(
                    f"  [OVERWRITE] Size mismatch — {_fname}\n"
                    f"              Existing file : {_dst_size:,} bytes\n"
                    f"              Incoming file : {_src_size:,} bytes\n"
                    f"              Action        : Overwriting with incoming file."
                )
                _overwrote += 1

        # ── Copy with full integrity verification ─────────────────────────
        print(f"  Importing: {_fname} ({_src_size:,} bytes)...")
        try:
            _copy_with_verification(_src_path, _dst_path)
            print(f"  [OK] Verified 100% — {_fname} ({_src_size:,} bytes, SHA-256 confirmed)")
            _imported += 1
        except RuntimeError as _copy_err:
            print(f"  [ERROR] Transfer failed for {_fname}: {_copy_err}")
            _failed += 1
            if os.path.exists(_dst_path):
                try:
                    os.unlink(_dst_path)
                    print(f"  [CLEANUP] Removed partial/corrupt file: {_fname}")
                except OSError:
                    pass

    print(
        f"\n[Drive Import] Complete.\n"
        f"  Imported  : {_imported} file(s) (100% verified)\n"
        f"  Skipped   : {_skipped} file(s) (identical duplicates)\n"
        f"  Overwrote : {_overwrote} file(s)\n"
        f"  Failed    : {_failed} file(s)\n"
        f"  Destination: {dest_dir}"
    )

    if _failed > 0:
        print(f"[Drive Import] WARNING: {_failed} file(s) failed transfer verification. "
              f"Check errors above.")

    return _imported


if AUTO_IMPORT_FROM_DRIVE:
    if not _DRIVE_MOUNTED:
        sys.exit(
            "[Drive Import] Cannot import from Drive — Drive is not mounted.\n"
            "Enable AUTO_MOUNT_DRIVE or mount Drive manually, then re-run."
        )
    _drive_audio_path = os.path.join(DRIVE_BASE, DRIVE_AUDIO_FOLDER.strip().lstrip("/"))
    print(f"[Drive Import] Source path: {_drive_audio_path}")
    _n_imported = _import_from_drive_folder(_drive_audio_path, BULK_DIR)

# ── Phase 1: Directory Check ───────────────────────────────────────────
# When drive import is ENABLED:  bulk dir was populated above — just validate.
# When drive import is DISABLED: standard create-and-prompt flow.

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(BULK_DIR):
    os.makedirs(BULK_DIR)
    sys.exit(
        f"\nDirectory created: '{BULK_DIR}'\n"
        f"Upload your audio/video files there, then run this cell again.\n"
    )

_all_in_dir = [
    os.path.join(BULK_DIR, f) for f in os.listdir(BULK_DIR)
    if os.path.isfile(os.path.join(BULK_DIR, f))
]
audio_files = sorted([
    f for f in _all_in_dir
    if os.path.splitext(f)[1].lower() in SUPPORTED_EXTENSIONS
])

if not audio_files:
    sys.exit(
        f"\nNo supported audio/video files found in '{BULK_DIR}'.\n"
        f"Supported formats: {', '.join(sorted(SUPPORTED_EXTENSIONS))}\n"
        f"Upload files there and run this cell again.\n"
    )

print(f"Found {len(audio_files)} file(s) to transcribe. Loading dependencies...")

# ── Phase 2: Full Imports ───────────────────────────────────────────────
# Only reached when files are confirmed present.

import contextlib
import gc
import subprocess
import tempfile
import threading
import time
import unicodedata
import zipfile

# ── Phase 3: Library Check / Install ───────────────────────────────────
# Only openai-whisper is needed; torch/numpy/tqdm are transitive dependencies.

def _install_libraries():
    cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "--root-user-action=ignore",
        "openai-whisper",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("[ERROR] pip install failed:")
        print(result.stderr)
        sys.exit(1)

try:
    import whisper
    import torch
    print("Libraries already installed. Skipping installation.")
except ImportError:
    print("Installing required libraries...")
    _install_libraries()
    try:
        import whisper
        import torch
        print("Libraries installed and imported successfully.")
    except ImportError as e:
        print(f"[ERROR] Import failed after installation: {e}")
        sys.exit(1)

# ── Phase 4: ffmpeg Binary Pre-flight Check ────────────────────────────

_ffmpeg_check = subprocess.run(
    ["ffmpeg", "-version"], capture_output=True
)
if _ffmpeg_check.returncode != 0:
    print("[ERROR] ffmpeg binary not found on PATH.")
    print("Fix: add a cell above with   !apt install -y ffmpeg   and run it first.")
    sys.exit(1)
print("ffmpeg binary confirmed.")

# ── Phase 4b: Environment Diagnostics ────────────────────────────────

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device         : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
print(f"PyTorch        : {torch.__version__}")

# ── Phase 5: Per-file Pre-validation ─────────────────────────────────

valid_audio_files = []
for _f in audio_files:
    _name = os.path.basename(_f)
    if not os.access(_f, os.R_OK):
        print(f"[SKIP] Not readable (check permissions): {_name}")
        continue
    if os.path.getsize(_f) == 0:
        print(f"[SKIP] Empty file (0 bytes): {_name}")
        continue
    valid_audio_files.append(_f)

if not valid_audio_files:
    sys.exit("[ERROR] No valid audio files remain after validation. Aborting.")

audio_files = valid_audio_files
print(f"{len(audio_files)} file(s) passed validation.")

# ── Phase 6: Model Load ───────────────────────────────────────────────

from whisper.utils import format_timestamp, get_writer  # noqa: E402

if DEVICE == "cpu":
    _total_secs    = 0.0
    _duration_known = True
    for _af in audio_files:
        _probe = subprocess.run(
            [
                "ffprobe", "-v", "error",
                "-show_entries", "format=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                _af,
            ],
            capture_output=True, text=True,
        )
        try:
            _total_secs += float(_probe.stdout.strip())
        except ValueError:
            _duration_known = False
            break

    _EST_SLOWDOWN = 20
    _border = "!" * 64
    print(f"\n{_border}")
    print("  WARNING: NO GPU DETECTED — TRANSCRIPTION WILL BE")
    print("  OVERWHELMINGLY SLOW (roughly 1 transcribed line per minute).")
    print()
    if _duration_known and _total_secs > 0:
        _audio_mins = _total_secs / 60
        _est_mins   = (_total_secs * _EST_SLOWDOWN) / 60
        _est_hrs    = _est_mins / 60
        print(f"  Total audio duration : ~{_audio_mins:.0f} min")
        if _est_hrs >= 1:
            print(f"  Estimated CPU time   : ~{_est_hrs:.1f} hours  (could be longer)")
        else:
            print(f"  Estimated CPU time   : ~{_est_mins:.0f} minutes  (could be longer)")
        print()
    print("  STRONGLY RECOMMENDED: switch to a GPU runtime first.")
    print("  Runtime > Change runtime type > Hardware accelerator > T4 GPU")
    print(f"{_border}\n")

    _answer = input(
        "Continue on CPU anyway? This will take an extremely long time. [y/N]: "
    ).strip().lower()

    if _answer != "y":
        sys.exit(
            "\nAborted. To enable GPU:\n"
            "  Runtime > Change runtime type > Hardware accelerator > T4 GPU\n"
            "Then run this cell again.\n"
        )
    print("Acknowledged. Continuing on CPU — this will take a very long time.\n")

# Guard: skip load if model already resident on the correct device.
# Re-running the cell retains `model` in the kernel namespace. Calling
# whisper.load_model() again double-allocates ~3 GB of VRAM before the
# old reference is GC'd, causing OOM or CUDA context corruption.
_need_load = True
if "model" in globals() and hasattr(model, "transcribe"):
    try:
        _loaded_device = str(next(model.parameters()).device).split(":")[0]
        if _loaded_device == DEVICE:
            print(f"Model '{USE_MODEL}' already loaded on {DEVICE}. Reusing.")
            _need_load = False
        else:
            print(
                f"Model is on '{_loaded_device}' but DEVICE is '{DEVICE}'. "
                "Releasing and reloading..."
            )
            del model
            gc.collect()
            # Justified: releasing a model from one device before reloading onto
            # a different device is a major phase transition. Clearing the cache
            # here prevents stale CUDA allocations from blocking the new load.
            # Per PyTorch guidance, this is a correct use of empty_cache().
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
    except Exception:
        print("Could not verify existing model state. Reloading to be safe.")
        try:
            del model
            gc.collect()
        except NameError:
            pass

if _need_load:
    print(f"Loading '{USE_MODEL}' model on {DEVICE}...")
    try:
        model = whisper.load_model(USE_MODEL, device=DEVICE)
        print(f"Model '{USE_MODEL}' loaded on {DEVICE}.")
    except Exception as e:
        print(f"[ERROR] Failed to load Whisper model: {e}")
        sys.exit(1)

# cuDNN autotuner — benchmark kernel implementations once, reuse fastest.
if CUDNN_BENCHMARK and DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    print("cuDNN benchmark mode : enabled")

# torch.compile() — optional experimental compilation via TorchDynamo/Inductor.
if TORCH_COMPILE:
    try:
        print("Compiling model with torch.compile() — warmup may take ~60s on first call...")
        model = torch.compile(model)
        print("Model compiled successfully.")
    except Exception as e:
        print(f"[WARN] torch.compile() failed (non-fatal, continuing without): {e}")

if DEVICE == "cuda":
    _vram_used  = torch.cuda.memory_allocated() / 1e9
    _vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {_vram_used:.2f} GB used / {_vram_total:.2f} GB total")

# ── Phase 7: Audio Preparation Helper ────────────────────────────────
# Note: LiveFileWriter has been removed — it was vestigial in this variant.
# It only mirrored sys.stdout and opened no file handle. The .txt transcript
# is written directly from result['segments'] after transcription completes.
# Verbose output from model.transcribe(verbose=True) goes straight to stdout.

def _is_already_optimal_wav(path):
    """Return True if the file is already a 16 kHz mono PCM WAV (no conversion needed)."""
    if os.path.splitext(path)[1].lower() != ".wav":
        return False
    _probe = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-select_streams", "a:0",
            "-show_entries", "stream=codec_name,sample_rate,channels",
            "-of", "default=noprint_wrappers=1:nokey=1",
            path,
        ],
        capture_output=True, text=True,
    )
    info = _probe.stdout.strip().split("\n")
    return (
        len(info) >= 3
        and info[0].strip() == "pcm_s16le"
        and info[1].strip() == "16000"
        and info[2].strip() == "1"
    )


def _prepare_audio(source_path, is_video):
    """Convert source_path to a 16 kHz mono WAV suitable for Whisper.
    Returns (tmp_path, converted) — if converted=False, source was already optimal.
    """
    if not is_video and _is_already_optimal_wav(source_path):
        return source_path, False

    tmp = tempfile.NamedTemporaryFile(
        suffix=".wav",
        prefix="_whisper_pcm_",
        delete=False,
    )
    tmp_path = tmp.name
    tmp.close()

    cmd = [
        "ffmpeg", "-y",
        "-i", source_path,
        "-vn",                   # discard video stream (no-op for pure audio)
        "-acodec", "pcm_s16le",  # 16-bit signed PCM — Whisper's native format
        "-ar", "16000",          # 16 kHz sample rate
        "-ac", "1",              # mono
        tmp_path,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass
        err = proc.stderr.strip().splitlines()[-1] if proc.stderr.strip() else "unknown ffmpeg error"
        raise RuntimeError(f"Audio preparation failed: {err}")

    return tmp_path, True


# ── Phase 8: Timestamped .txt Writer ──────────────────────────────────

def _write_timestamped_txt(segments, output_path):
    """Write a timestamped plain-text transcript from DTW-aligned Whisper segments."""
    with open(output_path, "w", encoding="utf-8") as f:
        for seg in segments:
            start = format_timestamp(seg["start"], always_include_hours=True)
            end   = format_timestamp(seg["end"],   always_include_hours=True)
            text  = seg["text"].strip()
            if text:
                f.write(f"[{start} --> {end}]  {text}\n")


# ── Phase 9: Transcription Options & Memory Management ─────────────────────
#
# ╔══════════════════════════════════════════════════════════════════════╗
#  PyTorch Memory Management — Per PyTorch Maintainer Guidance              ║
#  ║                                                                          ║
#  torch.cuda.empty_cache() releases unused memory from the INTERNAL        ║
#  CACHING ALLOCATOR back to the CUDA memory pool. It does NOT prevent      ║
#  CUDA OOM errors — the allocator already reuses cached memory           ║
#  automatically between operations.                                         ║
#  ║                                                                          ║
#  Calling empty_cache() in a per-file loop REDUCES performance: freed      ║
#  memory must be re-allocated with synchronizing cudaMalloc calls on the   ║
#  next iteration, adding latency and CUDA synchronization overhead.        ║
#  ║                                                                          ║
#  This implementation:                                                      ║
#    - Calls empty_cache() ONLY at major phase transitions (model reload     ║
#      across devices) — the one clearly justified use case.              ║
#    - Per-file cache clearing REMOVED from transcription loop.             ║
#    - OOM recovery uses: del result → gc.collect() → empty_cache()       ║
#      (appropriate: error recovery state, not a processing loop).          ║
#    - Explicit `del result` after each file frees the Python-side          ║
#      segment list from RAM; gc.collect() runs once after all files done.  ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# Two presets controlled by FAST_DECODE:
#
#   FAST_DECODE=False (default — quality mode):
#     beam_size=5 activates beam search. best_of=5 is intentionally kept for
#     compatibility but is ignored when beam_size > 1 (Whisper uses beam search
#     and disables stochastic sampling). temperature is a 6-step fallback retry
#     schedule — Whisper retries a segment with the next temperature value when
#     the output fails compression-ratio or logprob quality checks.
#
#   FAST_DECODE=True (speed mode):
#     Greedy decoding (beam_size=1, temperature=0.0). Fallback retries disabled
#     by setting all thresholds to None. Up to 2x faster on clean speech.

if FAST_DECODE:
    _options = {
        "task":                        "transcribe",
        "fp16":                        DEVICE == "cuda",
        # Greedy decoding — fastest possible decode path.
        # beam_size=1 disables beam search; temperature=0 disables sampling.
        "beam_size":                   1,
        "temperature":                 0.0,
        # Disable fallback retries — prevents repeated decode passes per segment.
        "compression_ratio_threshold": None,
        "logprob_threshold":           None,
        "no_speech_threshold":         0.6,
        "condition_on_previous_text":  True,
        "initial_prompt":              None,
        "language":                    "ja",
        # DTW word-level alignment — kept even in fast mode; runs during the
        # existing forward pass with minimal overhead (no extra decode pass).
        "word_timestamps":             True,
    }
    print("FAST_DECODE mode: greedy decoding, fallback retries disabled.")
else:
    _options = {
        "task":                        "transcribe",
        "fp16":                        DEVICE == "cuda",
        # beam_size=5 activates beam search (5 beams).
        "beam_size":                   5,
        # best_of is only active when beam_size is None or 1 (stochastic
        # sampling). With beam_size=5, best_of is silently ignored by Whisper's
        # decoder. Kept here for future compatibility if beam_size is changed.
        "best_of":                     5,
        # temperature is a fallback retry schedule: if a segment fails quality
        # checks (compression ratio, logprob), Whisper retries with the next
        # temperature. 0.0 = greedy first attempt, higher = more random.
        "temperature":                 (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
        "condition_on_previous_text":  True,
        "initial_prompt":              None,
        "language":                    None,
        "no_speech_threshold":         0.4,
        "logprob_threshold":           -1.5,
        "compression_ratio_threshold": 2.2,
        # Enables DTW cross-attention alignment (timing.py).
        # Overwrites seg['start']/seg['end'] with word-boundary-aligned
        # timestamps derived from which audio frames the model attended to.
        "word_timestamps":             True,
    }

# torch.inference_mode() context — disables autograd bookkeeping for a free
# ~3-10% speedup. contextlib.nullcontext() used when flag is off so
# the transcription call site is identical either way.
_infer_ctx = torch.inference_mode() if USE_INFERENCE_MODE else contextlib.nullcontext()
if USE_INFERENCE_MODE:
    print("inference_mode : enabled")

_mode_label = "FAST (greedy)" if FAST_DECODE else "QUALITY (beam search)"
print(f"Decode mode    : {_mode_label}")
print(f"Audio preconv  : {'enabled' if PRECONVERT_AUDIO else 'disabled'}")

# ── Phase 10: Transcription Loop ───────────────────────────────────────────

succeeded_paths = set()
failed_files    = {}
n = len(audio_files)

for i, audio_path in enumerate(audio_files, 1):
    raw_stem    = os.path.splitext(os.path.basename(audio_path))[0]
    output_stem = unicodedata.normalize("NFC", raw_stem)
    txt_path    = os.path.join(OUTPUT_DIR, output_stem + ".txt")

    print(f"\n[{i}/{n}] Processing: {os.path.basename(audio_path)}")

    _temp_audio     = None
    transcribe_path = audio_path
    _is_video       = os.path.splitext(audio_path)[1].lower() in VIDEO_EXTENSIONS

    if _is_video or PRECONVERT_AUDIO:
        _label = "Video — extracting audio" if _is_video else "Pre-converting to 16 kHz mono WAV"
        print(f"  {_label}...")
        try:
            _prepared, _converted = _prepare_audio(audio_path, is_video=_is_video)
            if _converted:
                _temp_audio     = _prepared
                transcribe_path = _prepared
                print(f"  Audio ready.")
            else:
                print(f"  Already 16 kHz mono WAV — skipping conversion.")
        except RuntimeError as e:
            msg = str(e)
            print(f"[FAIL] {os.path.basename(audio_path)}: {msg}")
            failed_files[audio_path] = msg
            continue

    # .txt is NOT pre-cleared — if transcription fails, the previous .txt is preserved.

    result = None
    _t0    = time.time()

    try:
        with _infer_ctx:
            result = model.transcribe(transcribe_path, verbose=True, **_options)

    except RuntimeError as e:
        msg = str(e)
        if "out of memory" in msg.lower():
            # OOM recovery: explicit cleanup before cache flush.
            # Per PyTorch guidance, empty_cache() here is appropriate:
            # we are in an error-recovery state, not a normal processing loop.
            # Order: del → gc.collect() → empty_cache().
            if result is not None:
                del result
                result = None
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            msg = (
                "GPU out of memory. "
                "Try a shorter audio file, or restart the runtime to free VRAM: "
                "Runtime > Restart session."
            )
        else:
            msg = f"RuntimeError: {e}"
        failed_files[audio_path] = msg
        print(f"\n[FAIL] {os.path.basename(audio_path)}: {msg}")

    except Exception as e:
        msg = str(e)
        failed_files[audio_path] = msg
        print(f"\n[FAIL] {os.path.basename(audio_path)}: {msg}")

    finally:
        if _temp_audio and os.path.exists(_temp_audio):
            os.unlink(_temp_audio)

    if result is None:
        continue

    # NOTE: torch.cuda.empty_cache() after each file has been REMOVED.
    # Per PyTorch maintainer guidance, calling empty_cache() in a per-file
    # loop forces re-allocation of GPU memory on each subsequent file via
    # synchronizing cudaMalloc calls. PyTorch's caching allocator handles
    # memory reuse automatically — no intervention needed between files.

    if not result.get("segments"):
        print(f"  No speech detected in: {os.path.basename(audio_path)}")

    succeeded_paths.add(audio_path)

    if result.get("segments"):
        _write_timestamped_txt(result["segments"], txt_path)
        print(f"  Saved: {output_stem}.txt")

    print(f"  Completed in {time.time() - _t0:.1f}s")

    norm_audio_path = os.path.join(
        os.path.dirname(audio_path),
        output_stem + os.path.splitext(audio_path)[1]
    )

    for fmt in ("srt", "vtt"):
        try:
            fix_vtt = (
                fmt == "vtt"
                and bool(result.get("segments"))
                and result["segments"][0].get("start") == 0
            )
            _orig_start = result["segments"][0]["start"] if fix_vtt else None
            if fix_vtt:
                result["segments"][0]["start"] += 1 / 1000

            try:
                writer = get_writer(fmt, OUTPUT_DIR)
                writer(result, norm_audio_path)
            finally:
                if fix_vtt:
                    result["segments"][0]["start"] = _orig_start

            print(f"  Saved: {output_stem}.{fmt}")

        except Exception as e:
            print(f"  [WARN] Could not write .{fmt} for '{output_stem}': {e}")

    # Explicit per-file cleanup: free the large transcription result dict
    # (Python-side segment list) from RAM before the next iteration.
    # PyTorch's CUDA allocator manages GPU memory automatically — del result
    # only affects the Python heap, not VRAM. gc.collect() is deferred to
    # after the full loop to avoid per-file overhead.
    del result
    result = None

# gc.collect() once after all files complete — cleans up any remaining
# Python-side references without per-file overhead.
gc.collect()

# ── Phase 11: Zip All Transcribed Files ───────────────────────────────────

if not succeeded_paths:
    print("\n[ERROR] No files transcribed successfully. Nothing to zip.")
else:
    _files_to_zip = sorted([
        os.path.join(OUTPUT_DIR, fname)
        for fname in os.listdir(OUTPUT_DIR)
        if os.path.splitext(fname)[1].lower() in TRANSCRIBED_EXTS
    ])

    if not _files_to_zip:
        print("\n[WARN] Output directory has no .txt/.srt/.vtt files to zip.")
    else:
        with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
            for fpath in _files_to_zip:
                zf.write(fpath, arcname=os.path.basename(fpath))

        _zip_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
        print(
            f"\nZipped {len(_files_to_zip)} file(s) -> {ZIP_PATH} "
            f"({_zip_mb:.2f} MB)"
        )
        print(
            "REMINDER: Download your zip before the Colab session ends. "
            "/content/ is erased when the runtime disconnects."
        )

# ── Phase 11b: Auto-Export Transcription Archive to Google Drive ───────────────
# Uploads ZIP_PATH to Drive after all transcription jobs complete and the zip
# has been created.
#
# Destination resolution (all fallbacks explicitly printed):
#   1. DRIVE_EXPORT_FOLDER if set.
#   2. DRIVE_AUDIO_FOLDER as fallback (explicitly printed).
#   3. Warning + skip if neither is configured.
#
# Collision policy: prompt y/n with 120s timeout, then upload with timestamp suffix.
# Export folder is auto-created if it does not exist.
# Upload verified via size + SHA-256 (same _copy_with_verification used for import).

if AUTO_EXPORT_TO_DRIVE and succeeded_paths and os.path.exists(ZIP_PATH):
    if not _DRIVE_MOUNTED:
        print(
            "[Drive Export] WARNING: AUTO_EXPORT_TO_DRIVE is enabled but Drive is not\n"
            "               mounted. Set AUTO_MOUNT_DRIVE=True and re-run to export.\n"
            f"               Your zip is available locally at: {ZIP_PATH}"
        )
    else:
        # ── Resolve export destination ───────────────────────────────────────────
        _export_folder       = None
        _export_source_label = None

        if DRIVE_EXPORT_FOLDER.strip():
            _export_folder       = os.path.join(DRIVE_BASE, DRIVE_EXPORT_FOLDER.strip().lstrip("/"))
            _export_source_label = f"DRIVE_EXPORT_FOLDER: '{DRIVE_EXPORT_FOLDER.strip()}'"
            print(f"[Drive Export] Destination  : {_export_folder}")
            print(f"[Drive Export] Source setting: {_export_source_label}")
        elif DRIVE_AUDIO_FOLDER.strip():
            _export_folder       = os.path.join(DRIVE_BASE, DRIVE_AUDIO_FOLDER.strip().lstrip("/"))
            _export_source_label = (
                f"Fallback — DRIVE_EXPORT_FOLDER is blank; "
                f"using DRIVE_AUDIO_FOLDER: '{DRIVE_AUDIO_FOLDER.strip()}'"
            )
            print(f"[Drive Export] DRIVE_EXPORT_FOLDER is blank.")
            print(f"[Drive Export] Fallback destination: {_export_folder}")
            print(f"[Drive Export] Reason: {_export_source_label}")
        else:
            print(
                "[Drive Export] WARNING: Neither DRIVE_EXPORT_FOLDER nor DRIVE_AUDIO_FOLDER\n"
                "               is configured. Cannot determine export destination.\n"
                "               Skipping Drive upload. Your zip is here:\n"
                f"               {ZIP_PATH}\n"
                "               Download it manually before the session ends."
            )

        if _export_folder is not None:
            # Auto-create export folder if it does not exist
            if not os.path.exists(_export_folder):
                os.makedirs(_export_folder, exist_ok=True)
                print(f"[Drive Export] Created folder: {_export_folder}")

            _zip_basename = os.path.basename(ZIP_PATH)
            _dst_path     = os.path.join(_export_folder, _zip_basename)

            # ── Collision: prompt y/n with 120s timeout then timestamp suffix ──
            if os.path.exists(_dst_path):
                _user_choice = [None]

                def _prompt_overwrite():
                    try:
                        _user_choice[0] = input(
                            f"\n[Drive Export] '{_zip_basename}' already exists at destination.\n"
                            f"  Path: {_dst_path}\n"
                            f"  Overwrite? [y/N] (auto-uploads with timestamp suffix in 120s): "
                        ).strip().lower()
                    except Exception:
                        _user_choice[0] = None

                _prompt_thread = threading.Thread(target=_prompt_overwrite, daemon=True)
                _prompt_thread.start()
                _prompt_thread.join(timeout=120)

                if _user_choice[0] == "y":
                    print("[Drive Export] Overwriting existing file.")
                else:
                    _ts            = time.strftime("%Y%m%d_%H%M%S")
                    _stem, _ext    = os.path.splitext(_zip_basename)
                    _zip_basename  = f"{_stem}_{_ts}{_ext}"
                    _dst_path      = os.path.join(_export_folder, _zip_basename)
                    if _user_choice[0] is None:
                        print(f"[Drive Export] No response in 120s — uploading as: {_zip_basename}")
                    else:
                        print(f"[Drive Export] Keeping existing — uploading as: {_zip_basename}")

            # ── Upload with full integrity verification ──────────────────────
            _zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
            print(f"[Drive Export] Uploading {os.path.basename(ZIP_PATH)} ({_zip_size_mb:.2f} MB)...")
            print(f"[Drive Export] Destination: {_dst_path}")
            try:
                _copy_with_verification(ZIP_PATH, _dst_path)
                _dst_mb = os.path.getsize(_dst_path) / (1024 * 1024)
                print(
                    f"[Drive Export] Upload complete — verified 100%.\n"
                    f"  Destination : {_dst_path}\n"
                    f"  Size        : {_dst_mb:.2f} MB (size + SHA-256 confirmed)"
                )
            except RuntimeError as _export_err:
                print(f"[Drive Export] ERROR: Verification failed — {_export_err}")
            except Exception as _export_err:
                print(f"[Drive Export] ERROR: Unexpected failure — {_export_err}")

elif AUTO_EXPORT_TO_DRIVE and not succeeded_paths:
    print("[Drive Export] Skipped — no files transcribed successfully.")
elif AUTO_EXPORT_TO_DRIVE and not os.path.exists(ZIP_PATH):
    print(f"[Drive Export] Skipped — zip file not found at {ZIP_PATH}.")

# ── Phase 12: Final Summary ───────────────────────────────────────────────

print("\n" + "=" * 52)
print("TRANSCRIPTION SUMMARY")
print(f"  Total files : {n}")
print(f"  Succeeded   : {len(succeeded_paths)}")
print(f"  Failed      : {len(failed_files)}")
if failed_files:
    print("\nFailed files:")
    for _path, _reason in failed_files.items():
        print(f"  x {os.path.basename(_path)}")
        print(f"    Reason: {_reason}")
print("=" * 52)


SystemExit: 
Directory created: '/content/bulk_process_audios_here'
Upload your audio/video files there, then run this cell again.


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
